# Can Neural GPUs Learn Arithmetic Algorithms & Generalize Beyond Training Length?

In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

## 1. Create a Binary-Addition Generator

In [2]:
FIXED_LENGTH = 8

class ValidateBinary(Exception):
  pass

class PaddingException(Exception):
  pass

def validate_binary(num: str):
  if num and set(num).issubset({'0', '1'}):
    return True
  raise ValidateBinary(
      "Binary numbers are only 0 and 1."
  )

def padding(num: str, length: int):
  if len(num) > length:
    raise PaddingException(
        f"Number must be at most {length} bits."
    )
  return num.zfill(length)

def binary_addition(a: str, b: str):
  validate_binary(a)
  validate_binary(b)

  a = padding(a, FIXED_LENGTH)
  b = padding(b, FIXED_LENGTH)

  result = bin(int(a, 2) + int(b, 2))[2:]

  return padding(result, FIXED_LENGTH + 1)

a = "00011111"
b = "1010101"
binary_addition(a, b)

'001110100'

### 1.1 Generator Many Binary Addition Samples

In [3]:
import random

def generate_sample(bit_length):
    max_value = 2**bit_length - 1

    a = random.randint(0, max_value)
    b = random.randint(0, max_value)

    a_binary = bin(a)[2:].zfill(bit_length)
    b_binary = bin(b)[2:].zfill(bit_length)

    result = bin(a + b)[2:].zfill(bit_length + 1)

    return a_binary, b_binary, result

a, b, result = generate_sample(FIXED_LENGTH)
print(a, b, result)

00111001 00001100 001000101


In [4]:
def generate_dataset(n_samples, bit_length):
    return [
        generate_sample(bit_length)
        for _ in range(n_samples)
    ]

dataset = generate_dataset(1000, FIXED_LENGTH)

dataset[:5]

[('10001100', '01111101', '100001001'),
 ('01110010', '01000111', '010111001'),
 ('00110100', '00101100', '001100000'),
 ('11011000', '00010000', '011101000'),
 ('00001111', '00101111', '000111110')]

### 1.2 Convert Each Binary String into Tensor

In [5]:
import torch

def binary_to_tensor(binary: str):
    return torch.tensor(
        [int(bit) for bit in binary[::-1]],
        dtype=torch.long
    )

a, b, result = generate_sample(FIXED_LENGTH)

a_tensor = binary_to_tensor(a)
b_tensor = binary_to_tensor(b)
result_tensor = binary_to_tensor(result)

print(a_tensor)
print(b_tensor)
print(result_tensor)

print(a_tensor.shape)
print(result_tensor.shape)

tensor([1, 1, 0, 1, 1, 1, 1, 0])
tensor([1, 1, 0, 0, 1, 1, 1, 0])
tensor([0, 1, 1, 1, 0, 1, 1, 1, 0])
torch.Size([8])
torch.Size([9])


In [6]:
x = torch.stack([a_tensor, b_tensor], dim=1)
print(x.shape)

print(x[0])

torch.Size([8, 2])
tensor([1, 1])


## 2. Create PyTorch Dataset

In [7]:
from torch.utils.data import Dataset

class BinaryAdditionDataset(Dataset):
    def __init__(self, n_samples, bit_length):
        self.data = generate_dataset(
            n_samples,
            bit_length
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        a, b, result = self.data[idx]

        a_tensor = binary_to_tensor(a)
        b_tensor = binary_to_tensor(b)
        y = binary_to_tensor(result)

        x = torch.stack(
            [a_tensor, b_tensor],
            dim=1
        )

        return x, y

In [8]:
dataset = BinaryAdditionDataset(
    1000,
    bit_length=FIXED_LENGTH
)
x, y = dataset[0]

print(x)
print(y)

print(f"x shape is {x.shape}")
print(f"y shape is {y.shape}")

tensor([[1, 1],
        [0, 0],
        [0, 0],
        [1, 1],
        [1, 0],
        [0, 0],
        [0, 1],
        [1, 0]])
tensor([0, 1, 0, 0, 0, 1, 1, 1, 0])
x shape is torch.Size([8, 2])
y shape is torch.Size([9])


### 2.1 Create a DataLoader

In [9]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

x_batch, y_batch = next(iter(loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([32, 8, 2])
torch.Size([32, 9])


## 3. Construct Neural GPU initial State

In [10]:
import torch.nn as nn

NUM_INTERNAL_FEATURES = 16

input_projection = nn.Linear(2, NUM_INTERNAL_FEATURES)

x_batch = x_batch.float()
embedded = input_projection(x_batch)

print(embedded.shape)

torch.Size([32, 8, 16])


In [11]:
B = x_batch.shape[0]

s0 = torch.zeros(
    B,
    NUM_INTERNAL_FEATURES,
    1,
    FIXED_LENGTH + 1
)

print(f"s0 shape is {s0.shape}")

s0 shape is torch.Size([32, 16, 1, 9])


In [12]:
embedded = embedded.permute(0, 2, 1)

print(embedded.shape)

torch.Size([32, 16, 8])


## 4. Create CGRU Cell

In [13]:
class CGRU(nn.Module):
  def __init__(self, channels, kernel_size=3):
    super().__init__()

    padding = kernel_size // 2

    self.update_conv = nn.Conv2d(
        channels,
        channels,
        kernel_size,
        padding=padding
    )

    self.reset_conv = nn.Conv2d(
        channels,
        channels,
        kernel_size,
        padding=padding
    )

    self.candidate_conv = nn.Conv2d(
        channels,
        channels,
        kernel_size,
        padding=padding
    )

  def forward(self, s):

    u = torch.sigmoid(
        self.update_conv(s)
    )

    r = torch.sigmoid(
        self.reset_conv(s)
    )

    candidate = torch.tanh(
        self.candidate_conv(
            r * s
        )
    )

    s_next = (
        u * s
        + (1 - u) * candidate
    )

    return s_next

### 4.1 Test Shape THrough the Network

In [14]:
cgru = CGRU(channels=16)

s1 = cgru(s0)

print(s0.shape)
print(s1.shape)

torch.Size([32, 16, 1, 9])
torch.Size([32, 16, 1, 9])


In [15]:
NUM_STEPS = 8

s = s0

for _ in range(NUM_STEPS):
    s = cgru(s)

print(s.shape)

torch.Size([32, 16, 1, 9])


## 5. Create Neural GPU Network

In [16]:
class NeuralGPU(nn.Module):
    def __init__(self, m=16):
        super().__init__()

        self.m = m

        self.input_projection = nn.Linear(2, m)

        self.cgru = CGRU(
            channels=m,
            kernel_size=3
        )

        self.decoder = nn.Conv2d(
            in_channels=m,
            out_channels=2,
            kernel_size=1
        )

    def forward(self, x):
        L = x.shape[1]
        # x: [B, L, 2]

        embedded = self.input_projection(x.float())
        # [B, L, 16]

        embedded = embedded.permute(0, 2, 1)
        # [B, 16, L]

        B = x.shape[0]

        s = torch.zeros(
            B,
            self.m,
            1,
            L + 1,
            device=x.device
        )

        s[:, :, 0, :L] = embedded

        # Recurrent computation
        for _ in range(L):
            s = self.cgru(s)

        # Decode final state
        logits = self.decoder(s)
        # [B, 2, 1, 9]

        logits = logits.squeeze(2)
        # [B, 2, 9]

        logits = logits.permute(0, 2, 1)
        # [B, 9, 2]

        return logits

In [17]:
model = NeuralGPU(m=16)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [18]:
logits = model(x_batch)

print(logits.shape)
print(y_batch.shape)

torch.Size([32, 9, 2])
torch.Size([32, 9])


In [19]:
loss = criterion(
    logits.reshape(-1, 2),
    y_batch.reshape(-1)
)

## 6. Create Training Loop

In [20]:
EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x_batch, y_batch in loader:

        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(
            logits.reshape(-1, 2),
            y_batch.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/20 | Loss: 0.6937
Epoch 2/20 | Loss: 0.6928
Epoch 3/20 | Loss: 0.6908
Epoch 4/20 | Loss: 0.6858
Epoch 5/20 | Loss: 0.6742
Epoch 6/20 | Loss: 0.6244
Epoch 7/20 | Loss: 0.5065
Epoch 8/20 | Loss: 0.3377
Epoch 9/20 | Loss: 0.1915
Epoch 10/20 | Loss: 0.1061
Epoch 11/20 | Loss: 0.0635
Epoch 12/20 | Loss: 0.0367
Epoch 13/20 | Loss: 0.0262
Epoch 14/20 | Loss: 0.0209
Epoch 15/20 | Loss: 0.0157
Epoch 16/20 | Loss: 0.0169
Epoch 17/20 | Loss: 0.0096
Epoch 18/20 | Loss: 0.0096
Epoch 19/20 | Loss: 0.0055
Epoch 20/20 | Loss: 0.0074


In [21]:
model.eval()

correct_bits = 0
total_bits = 0

correct_sequences = 0
total_sequences = 0

with torch.no_grad():
    for x_batch, y_batch in loader:

        logits = model(x_batch)

        predictions = logits.argmax(dim=-1)

        # Bit accuracy
        correct_bits += (predictions == y_batch).sum().item()
        total_bits += y_batch.numel()

        # Exact-match accuracy
        sequence_correct = (predictions == y_batch).all(dim=1)

        correct_sequences += sequence_correct.sum().item()
        total_sequences += y_batch.size(0)

bit_accuracy = correct_bits / total_bits
exact_accuracy = correct_sequences / total_sequences

print(f"Bit Accuracy: {bit_accuracy:.4f}")
print(f"Exact Match Accuracy: {exact_accuracy:.4f}")

Bit Accuracy: 0.9992
Exact Match Accuracy: 0.9950


In [22]:
test_dataset = BinaryAdditionDataset(1000, FIXED_LENGTH)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [23]:
model.eval()

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        logits = model(x_batch)
        predictions = logits.argmax(dim=-1)

In [24]:
model.eval()

correct_bits = 0
total_bits = 0

correct_sequences = 0
total_sequences = 0

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        logits = model(x_batch)

        predictions = logits.argmax(dim=-1)

        correct_bits += (predictions == y_batch).sum().item()
        total_bits += y_batch.numel()

        exact = (predictions == y_batch).all(dim=1)

        correct_sequences += exact.sum().item()
        total_sequences += y_batch.size(0)


bit_accuracy = correct_bits / total_bits
exact_accuracy = correct_sequences / total_sequences

print(f"Bit Accuracy: {bit_accuracy:.4f}")
print(f"Exact Match Accuracy: {exact_accuracy:.4f}")

Bit Accuracy: 0.9982
Exact Match Accuracy: 0.9870


In [25]:
def evaluate(model, bit_length, n_samples=1000):

    test_dataset = BinaryAdditionDataset(
        n_samples,
        bit_length=bit_length
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    model.eval()

    correct_bits = 0
    total_bits = 0

    correct_sequences = 0
    total_sequences = 0

    with torch.no_grad():

        for x_batch, y_batch in test_loader:

            logits = model(x_batch)
            predictions = logits.argmax(dim=-1)

            correct_bits += (
                predictions == y_batch
            ).sum().item()

            total_bits += y_batch.numel()

            exact = (
                predictions == y_batch
            ).all(dim=1)

            correct_sequences += exact.sum().item()
            total_sequences += y_batch.size(0)

    bit_accuracy = correct_bits / total_bits
    exact_accuracy = correct_sequences / total_sequences

    return bit_accuracy, exact_accuracy

In [26]:
test_lengths = [8, 10, 12, 16, 24, 32]

bit_accuracies = []
exact_accuracies = []

for length in test_lengths:

    bit_acc, exact_acc = evaluate(
        model,
        bit_length=length
    )

    bit_accuracies.append(bit_acc)
    exact_accuracies.append(exact_acc)

    print(
        f"{length} bits | "
        f"Bit Accuracy: {bit_acc:.4f} | "
        f"Exact Match: {exact_acc:.4f}"
    )

8 bits | Bit Accuracy: 0.9991 | Exact Match: 0.9940
10 bits | Bit Accuracy: 0.9976 | Exact Match: 0.9840
12 bits | Bit Accuracy: 0.9868 | Exact Match: 0.8560
16 bits | Bit Accuracy: 0.9137 | Exact Match: 0.1860
24 bits | Bit Accuracy: 0.7702 | Exact Match: 0.0040
32 bits | Bit Accuracy: 0.7382 | Exact Match: 0.0000
